# Construction PPE Detector
A simple end-to-end notebook: check the GPU, download the public dataset, train YOLO, evaluate the test split, and export the model. In Colab choose **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable GPU from Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics==8.3.221
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
from collections import Counter
from importlib.metadata import files as package_files
import hashlib, json, shutil
import numpy as np

## Train or evaluate the existing checkpoint
Ultralytics downloads the Construction-PPE dataset automatically. It contains 1,416 images split into train, validation, and test sets, with 11 PPE-related classes. Keep `USE_EXISTING_CHECKPOINT = False` for a new YOLO11n training run. Set it to `True` to upload and evaluate the repository's current `models/best.pt` without retraining.

In [ ]:
USE_EXISTING_CHECKPOINT = False
run_dir = Path('/content/runs/ppe-yolo11n')
if run_dir.exists():
    shutil.rmtree(run_dir)

if USE_EXISTING_CHECKPOINT:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload exactly one best.pt checkpoint'
    uploaded_name, checkpoint_bytes = next(iter(uploaded.items()))
    assert uploaded_name.lower().endswith('.pt'), 'Checkpoint must be a .pt file'
    best_path = run_dir / 'weights/best.pt'
    best_path.parent.mkdir(parents=True, exist_ok=True)
    best_path.write_bytes(checkpoint_bytes)
    print('Using uploaded checkpoint:', uploaded_name)
else:
    model = YOLO('yolo11n.pt')
    model.train(
        data='construction-ppe.yaml',
        epochs=40,
        imgsz=640,
        batch=16,
        device=0,
        patience=10,
        project='/content/runs',
        name='ppe-yolo11n',
        exist_ok=True,
        seed=42,
    )

## Evaluate on the test split
These are the numbers to discuss in an interview. They come from images that were not used for training or early stopping.

In [ ]:
best_path = run_dir / 'weights/best.pt'
assert best_path.exists(), f'Training checkpoint not found: {best_path}'
best_model = YOLO(best_path)
check_det_dataset('construction-ppe.yaml')
dataset_root = Path('/content/datasets/construction-ppe')
test_images = sorted((dataset_root / 'images/test').glob('*'))
assert test_images, 'Could not find test images'
eval_dir = Path('/content/evaluation/test')
if eval_dir.exists():
    shutil.rmtree(eval_dir)
metrics = best_model.val(
    data='construction-ppe.yaml', split='test', imgsz=640, device=0, plots=True,
    conf=0.001, iou=0.70,
    project=str(eval_dir.parent), name=eval_dir.name, exist_ok=True,
)
summary = {
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50_95': round(float(metrics.box.map), 4),
}
print(json.dumps(summary, indent=2))
metrics_path = eval_dir / 'metrics.json'
metrics_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

# Count training instances so class-imbalance claims are based on labels.
train_images = sorted((dataset_root / 'images/train').glob('*'))
all_train_labels = sorted((dataset_root / 'labels/train').glob('*.txt'))
assert train_images and all_train_labels, 'Could not find the training split'
train_image_stems = {path.stem for path in train_images}
train_label_stems = {path.stem for path in all_train_labels}
train_labels = [path for path in all_train_labels if path.stem in train_image_stems]
orphan_train_labels = [path for path in all_train_labels if path.stem not in train_image_stems]
train_images_without_labels = [
    path for path in train_images if path.stem not in train_label_stems
]
train_counts = Counter()
for label_path in train_labels:
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            train_counts[int(line.split()[0])] += 1
class_distribution = {
    'split': 'train',
    'images': len(train_images),
    'matched_label_files': len(train_labels),
    'orphan_label_files_ignored': len(orphan_train_labels),
    'images_without_label_files': len(train_images_without_labels),
    'instances': sum(train_counts.values()),
    'classes': {
        best_model.names[class_id]: {
            'class_id': class_id,
            'instances': train_counts.get(class_id, 0),
        }
        for class_id in sorted(best_model.names)
    },
}
class_distribution_path = eval_dir / 'train_class_distribution.json'
class_distribution_path.write_text(
    json.dumps(class_distribution, indent=2), encoding='utf-8'
)
distribution_table = [
    '| Class | Training instances |',
    '|---|---:|',
]
for class_name, values in class_distribution['classes'].items():
    distribution_table.append(f"| {class_name} | {values['instances']} |")
class_distribution_table_path = eval_dir / 'train_class_distribution.md'
distribution_table.extend([
    '',
    f"Matched label files: {len(train_labels)}; "
    f"orphan label files ignored: {len(orphan_train_labels)}; "
    f"images without label files: {len(train_images_without_labels)}.",
])
class_distribution_table_path.write_text(
    '\n'.join(distribution_table) + '\n', encoding='utf-8'
)
print('\n'.join(distribution_table))

# Sample the test-set confidence curves at explicit deployment thresholds.
# Ultralytics builds these curves from class-aware matches at IoU 0.50.
fixed_summary = {
    'split': 'test',
    'images': len(test_images),
    'imgsz': 640,
    'match_iou': 0.50,
    'nms_iou': 0.70,
    'confidence_floor': 0.001,
    'averaging': 'macro over classes with test labels',
    'thresholds': {},
}
for threshold in (0.20, 0.35, 0.50):
    per_class = {}
    precision_values, recall_values, f1_values = [], [], []
    for row, class_id in enumerate(metrics.box.ap_class_index):
        class_id = int(class_id)
        precision = float(np.interp(threshold, metrics.box.px, metrics.box.p_curve[row]))
        recall = float(np.interp(threshold, metrics.box.px, metrics.box.r_curve[row]))
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        precision_values.append(precision)
        recall_values.append(recall)
        f1_values.append(f1)
        per_class[metrics.names[class_id]] = {
            'class_id': class_id,
            'support': int(metrics.nt_per_class[class_id]),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
        }
    fixed_summary['thresholds'][f'{threshold:.2f}'] = {
        'precision_macro': round(float(np.mean(precision_values)), 4),
        'recall_macro': round(float(np.mean(recall_values)), 4),
        'f1_macro': round(float(np.mean(f1_values)), 4),
        'per_class': per_class,
    }
fixed_metrics_path = eval_dir / 'fixed_threshold_metrics.json'
fixed_metrics_path.write_text(json.dumps(fixed_summary, indent=2), encoding='utf-8')
threshold_table = [
    '| Confidence | Precision | Recall | F1 |',
    '|---:|---:|---:|---:|',
]
for threshold, values in fixed_summary['thresholds'].items():
    threshold_table.append(
        f"| {threshold} | {values['precision_macro']:.4f} | "
        f"{values['recall_macro']:.4f} | {values['f1_macro']:.4f} |"
    )
fixed_table_path = eval_dir / 'fixed_threshold_metrics.md'
fixed_table_path.write_text('\n'.join(threshold_table) + '\n', encoding='utf-8')
print('\n'.join(threshold_table))

## Preview predictions

In [ ]:
prediction_dir = Path('/content/predictions/examples')
if prediction_dir.exists():
    shutil.rmtree(prediction_dir)
best_model.predict(
    test_images[:6], conf=0.35, save=True,
    project=str(prediction_dir.parent), name=prediction_dir.name, exist_ok=True,
)

## Export and download results

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
release = Path('/content/construction-ppe-results')
if release.exists():
    shutil.rmtree(release)
release.mkdir(parents=True)
shutil.copy2(best_path, release / 'best.pt')
shutil.copy2(onnx_path, release / 'best.onnx')
model_checksums = []
for model_name in ('best.pt', 'best.onnx'):
    model_file = release / model_name
    digest = hashlib.sha256(model_file.read_bytes()).hexdigest()
    model_checksums.append(f'{digest}  {model_name}')
(release / 'model_checksums.txt').write_text(
    '\n'.join(model_checksums) + '\n', encoding='utf-8'
)
shutil.copy2(metrics_path, release / 'metrics.json')
shutil.copy2(fixed_metrics_path, release / 'fixed_threshold_metrics.json')
shutil.copy2(fixed_table_path, release / 'fixed_threshold_metrics.md')
shutil.copy2(class_distribution_path, release / 'train_class_distribution.json')
shutil.copy2(class_distribution_table_path, release / 'train_class_distribution.md')
shutil.copytree(prediction_dir, release / 'prediction_examples')
license_file = next(
    path for path in package_files('ultralytics')
    if str(path).replace('\\', '/').endswith('licenses/LICENSE')
)
shutil.copy2(license_file.locate(), release / 'LICENSE')
for name in ('confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    source = eval_dir / name
    if source.exists(): shutil.copy2(source, release / name)
zip_path = Path('/content/construction-ppe-results.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive('/content/construction-ppe-results', 'zip', release)
print('Created /content/construction-ppe-results.zip')

In [ ]:
from google.colab import files
files.download('/content/construction-ppe-results.zip')